# 01 - Exploratory data analysis (INCLUDE-50)

Run from the repository root (`jupyter lab` there). Needs `processed/splits/` (from `python -m src.data.split`);
the keypoint part also needs `processed/keypoints/`.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from src.data.eda import run_eda
from src.utils.config import load_config, resolve_paths

cfg = load_config(ROOT / "configs" / "data.yaml")
summary = run_eda(cfg)          # writes results/eda/ and returns the summary
print(json.dumps({k: v for k, v in summary.items() if k != "categories"}, indent=2))

## Size and class balance
One row per class and split. The split is stratified, so every class has videos in train, val and test.

In [ ]:
eda_dir = resolve_paths(cfg)["results_root"] / "eda"
counts = pd.read_csv(eda_dir / "class_counts.csv")
display(counts.describe().loc[["min", "max", "mean"]].round(1))
display(Image(filename=str(eda_dir / "class_balance.png")))

## Categories
INCLUDE groups its signs into categories (adjectives, animals, ...).

In [ ]:
pd.Series(summary["categories"]).sort_values().plot.barh(figsize=(6, 4), color="#2a78d6");

## MediaPipe detection (Pipeline B input quality)
Share of frames in which the body and each hand were found. Hands are found much less often than the body,
which limits how well the pose models can read hand shape.

In [ ]:
if (eda_dir / "keypoint_detection.csv").exists():
    print(summary["keypoint_detection"])
    display(Image(filename=str(eda_dir / "keypoint_detection.png")))
else:
    print("No keypoints found: run src.features.extract_keypoints or copy processed/keypoints.")

## Can the test accuracy be trusted? The split leak check
INCLUDE videos are named `MVI_<number>`. A signer records a sign several times in a row, so consecutive numbers
of one class are near-duplicate takes. Our split is random per video, so takes of one session end up in
train and in test. The check below counts test videos that have a same-class training video within 2 file
numbers. It is a heuristic (the dataset has no signer id), but a high share means the test accuracy is
optimistic compared with a split by recording session.

In [ ]:
print(json.dumps(summary["split_leak_check"], indent=2))